# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print("Version:", metadata.version)
print("Published:", metadata.date_published)
print("Identifier:", metadata.identifier)
print("Citation:", getattr(metadata, 'cite_as', ''))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all available record sets and fields using their `@id`s.

In [ ]:
# List all record set IDs
for rs in metadata.record_sets:
    print(f"RecordSet: @id={rs.id} | name='{rs.name}' | description='{rs.description}'")
    print("  Columns/Fields:")
    for fld in rs.fields:
        print(f"    Field: @id={fld.id} | name='{fld.name}' | data_type={fld.data_type}")
    print()

# Store all record set ids for later use
record_set_ids = [rs.id for rs in metadata.record_sets]

# If no record sets exist, show warning
if not record_set_ids:
    print("No record sets found in the metadata. Check if the dataset contains tabular data.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for record set: {record_set_id}")
    # Load records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  [!] Empty record set: {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Sample data:")
    display(df.head(2))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers or grouping data by key attributes.

Below, we pick the first available record set and a numeric field (`@id`) for demonstration.

In [ ]:
from IPython.display import display

# Choose the first available record set
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    # Try to automatically select a numeric field
    rs_md = next(rs for rs in metadata.record_sets if rs.id == record_set_id)
    numeric_field = None
    for f in rs_md.fields:
        if f.data_type in ["Float", "Integer", "Number"]:
            # Prefer normalized abundance or protein intensity, common in MS data
            if f.name.lower().find("abundance") >= 0 or f.name.lower().find("intensity") >= 0:
                numeric_field = f.id
                break
            # Else take any numeric field
            if not numeric_field:
                numeric_field = f.id
    print(f"Using example record set: {record_set_id}")
    print(f"Using numeric field: {numeric_field}")

    # Filtering threshold
    threshold = 10
    if numeric_field and numeric_field in df.columns:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (showing up to 5 rows):")
        display(filtered_df.head())
        # Normalize
        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
        print(f"First 5 normalized values for {numeric_field}:")
        display(filtered_df[[numeric_field, normalized_col]].head())

        # Try grouping (use first string field that is not the numeric field)
        group_field = None
        for f in rs_md.fields:
            if (f.data_type == "Text" or f.data_type == "String") and f.id in df.columns and f.id != numeric_field:
                group_field = f.id
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"Mean {numeric_field} grouped by {group_field} (first five groups):")
            display(grouped_df.head())
    else:
        print("No numeric field available for EDA in this record set.")
else:
    print("No available record sets to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we'll plot a histogram for the chosen numeric field and, if grouping field is available, a boxplot grouped by that field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot grouped, if available
    if 'group_field' in locals() and group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        # For large group counts, limit to top 10
        group_counts = df[group_field].value_counts()
        top_groups = group_counts.nlargest(10).index
        mask = df[group_field].isin(top_groups)
        sns.boxplot(x=df.loc[mask, group_field], y=df.loc[mask, numeric_field])
        plt.title(f"Distribution of {numeric_field} by {group_field} (top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset using ML Croissant. The steps showed how to load metadata, extract record sets by their `@id`, perform basic EDA including filtering and normalization on numeric fields, and visualize data distributions. For further investigation, you may wish to:
- Explore more record sets, if present
- Examine relationships between different protein attributes or samples
- Build machine learning models on the preprocessed data

Refer to the dataset documentation and Croissant schema for field meanings and further research possibilities.